# Redução de Dimensionalidade das Métricas Locais via PCA

In [237]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import plotly.express as px
import plotly.io as pio

pio.templates.default = "plotly_dark"


### Carregando o dataset

In [238]:
DATASET_PATH = "../data/processed/dataset_features.parquet"

df_dataset = pd.read_parquet(DATASET_PATH)
df_dataset.shape
df_filtered = df_dataset.columns[df_dataset.columns.str.contains("age", case=False)]
df_filtered



Index(['Age'], dtype='object')

In [239]:
df_dataset.head()

,degree_0,degree_1,degree_2,degree_3,degree_4,degree_5,degree_6,degree_7,degree_8,degree_9,...,SKID_Diagnoses 1,SKID_Diagnoses 2,Comments_SKID_assessment,Hamilton_Scale,BSL23_sumscore,BSL23_behavior,AUDIT,Standard_Alcoholunits_Last_28days,Alcohol_Dependence_In_1st-3rd_Degree_relative,Relationship_Status
0,39449.859168,39102.152342,40387.630997,37566.960373,39408.112994,38719.391893,40815.072973,41275.499305,39649.516506,40196.211832,...,"#CODE:296.26 #DESC:Major Depressive Disorder,...",None,past subclinical dperession,0.0,7.0,1.0,7.0,29.5,No,No
1,41513.435372,41327.513954,40750.435276,40491.263384,41233.158432,41146.017954,40730.430082,41351.132165,39371.224746,39566.056291,...,#CODE:V71.09 #DESC:No Diagnosis or Condition ...,None,None,4.0,6.0,2.0,11.0,53.5,Yes (9),No
2,40727.834359,40001.600624,40267.149863,39308.151269,39753.137398,39336.794272,40117.319684,40446.206489,38441.008195,38531.272048,...,#CODE:V71.09 #DESC:No Diagnosis or Condition ...,None,None,4.0,6.0,2.0,11.0,53.5,Yes (9),No
3,38401.305380,38333.117784,37902.877026,37959.206795,38497.084816,37761.467224,37754.236560,37956.444283,36880.581031,37203.264413,...,#CODE:V71.09 #DESC:No Diagnosis or Condition ...,None,None,4.0,3.0,1.0,3.0,5,No,Yes
4,42558.276311,42950.141781,42023.976342,40857.941940,40478.291440,41090.310476,42519.571111,42339.443250,40755.136885,42859.521462,...,#CODE:V71.09 #DESC:No Diagnosis or Condition ...,None,None,4.0,3.0,1.0,3.0,5,No,Yes


### Seleção Métricas Locais

In [240]:
NODE_FEATURE_PREFIXES = [
    "degree_",
    "clustering_",
    "betweenness_",
    "local_efficiency_",
    "hub_frequency_",
    "weighted_degree_"
]

NODE_FEATURES = [
    c for c in df_dataset.columns
    if any(c.startswith(p) for p in NODE_FEATURE_PREFIXES)
]

NODE_GROUPS = {
    "degree": [c for c in df_dataset.columns if c.startswith("degree_")],
    "clustering": [c for c in df_dataset.columns if c.startswith("clustering_")],
    "betweenness": [c for c in df_dataset.columns if c.startswith("betweenness_")],
    "local_efficiency": [c for c in df_dataset.columns if c.startswith("local_efficiency_")],
    "weighted_degree": [c for c in df_dataset.columns if c.startswith("weighted_degree_")]
}


len(NODE_FEATURES)


366

In [241]:
X = df_dataset[NODE_FEATURES].values
y = df_dataset["condition"].values

### Padronização das Feaures

Substituindo Nan por zero:

In [242]:
X = np.nan_to_num(X, nan=0.0)

In [243]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [244]:
X_scaled.mean(axis=0)[:5], X_scaled.std(axis=0)[:5]

(array([-1.99705572e-15, -1.66466167e-15, -1.47491447e-15,  1.61486985e-16,
         9.66230463e-16]),
 array([1., 1., 1., 1., 1.]))

### PCA exploratório

In [245]:
np.isnan(X).sum()


np.int64(0)

In [246]:
nan_per_feature = pd.Series(np.isnan(X).sum(axis=0), index=NODE_FEATURES)
nan_per_feature[nan_per_feature > 0].sort_values(ascending=False)


Series([], dtype: int64)

In [247]:
pca_full = PCA()
X_pca_full = pca_full.fit_transform(X_scaled)

explained_var = pca_full.explained_variance_ratio_
cum_var = np.cumsum(explained_var)


In [248]:
df_var = pd.DataFrame({
    "component": np.arange(1, len(explained_var) + 1),
    "explained_variance": explained_var,
    "cumulative_variance": cum_var
})


In [249]:
df_var

,component,explained_variance,cumulative_variance
0,1,5.245205e-01,0.524521
1,2,8.573967e-02,0.610260
2,3,3.187226e-02,0.642132
3,4,2.968941e-02,0.671822
4,5,2.564496e-02,0.697467
...,...,...,...
160,161,6.133566e-09,1.000000
161,162,3.894672e-09,1.000000
162,163,2.612905e-09,1.000000
163,164,1.738400e-09,1.000000


In [250]:
fig = px.line(
    df_var,
    x="component",
    y="cumulative_variance",
    markers=True,
    title="Variância Acumulada Explicada pelo PCA"
)
fig.add_hline(y=0.8, line_dash="dash", annotation_text="80%")
fig.add_hline(y=0.9, line_dash="dash", annotation_text="90%")
fig.show()


Observa-se que aproximadamente 14 componentes são suficientes para explicar 80% da variância total.

### PCA com 2 componentes

In [251]:
NODE_FEATURE_PREFIXES = [
    "degree_",
    "clustering_",
    "betweenness_",
    "local_efficiency_",
    "hub_frequency_",
    "weighted_degree_"
]

# Create a dictionary where keys are the metric names (e.g., 'degree')
NODE_FEATURES_BY_TYPE = {
    prefix.rstrip("_"): [c for c in NODE_FEATURES if c.startswith(prefix)]
    for prefix in NODE_FEATURE_PREFIXES
}

In [252]:
def plot_pca_features(df, features, title="PCA"):
    # Filter data
    X = df[features].values
    X = np.nan_to_num(X, nan=0.0)
    
    # Scale
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # PCA
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_scaled)
    
    # Create DataFrame for plotting
    df_pca = pd.DataFrame(X_pca, columns=["PC1", "PC2"])
    
    # Add condition for coloring if available
    if "condition" in df.columns:
        df_pca["condition"] = df["condition"].values
        color_col = "condition"
    else:
        color_col = None
        
    # Plot
    fig = px.scatter(
        df_pca, x="PC1", y="PC2", 
        color=color_col,
        title=title,
        opacity=0.8
    )
    fig.show()
    return pca


In [253]:
pca_degree = plot_pca_features(
    df_dataset, # Replace with df_dataset if that's your variable name
    NODE_FEATURES_BY_TYPE["degree"],
    title="PCA das Métricas de Grau (Nós)"
)

PCA clustering

In [254]:
pca_clustering = plot_pca_features(
    df_dataset, # Replace with df_dataset if that's your variable name
    NODE_FEATURES_BY_TYPE["clustering"],
    title="PCA das Métricas de Grau (Nós)"
)

Centralidade de Intermediação

In [255]:
pca_betweenness = plot_pca_features(
    df_dataset,
    NODE_GROUPS["betweenness"],
    title="PCA da Centralidade de Intermediação (Nós)"
)


Grau ponderado

In [256]:
pca_wdeg = plot_pca_features(
    df_dataset,
    NODE_GROUPS["weighted_degree"],
    title="PCA do Grau Ponderado (Nós)"
)


-----

Todas as variáveis

In [257]:
pca_2 = PCA(n_components=2)
X_pca_2 = pca_2.fit_transform(X_scaled)

df_pca = pd.DataFrame(
    X_pca_2,
    columns=["PC1", "PC2"]
)
df_pca["condition"] = y


In [258]:
fig = px.scatter(
    df_pca,
    x="PC1",
    y="PC2",
    color="condition",
    title="Projeção PCA das Métricas Locais (2 Componentes)",
    opacity=0.8
)
fig.show()


Aqui você responde visualmente:

existe separação EO × EC? Não há separação clara entre EO e EC

existe sobreposição? Sim

clusters? Existem regiões de maior densidade, mas não clusters bem definidos.

A projeção das métricas locais no espaço definido pelos dois primeiros
componentes principais não evidenciou uma separação clara entre as
condições EO e EC. Observa-se uma ampla sobreposição entre os pontos
correspondentes às duas condições experimentais, indicando que os
padrões dominantes de variabilidade capturados pelos componentes
principais são compartilhados entre os grupos.

Embora seja possível identificar regiões de maior densidade de pontos,
não foram observados clusters bem definidos ou fronteiras visuais que
permitam distinguir claramente as condições experimentais no espaço
PC1–PC2.


Age

In [259]:
age_cols = df_dataset.columns[df_dataset.columns.str.contains("gen", case=False)]
age_cols

Index(['Gender_ 1=female_2=male'], dtype='object')

In [260]:
df_dataset = df_dataset.rename(columns={
    "Gender_ 1=female_2=male": "gender",
    "Age": "age",
    "Handedness": "handedness",
    "Education": "education",
    "DRUG_0=negative_1=Positive": "drug",
    "Smoking_num_(Non-smoker=1, Occasional Smoker=2, Smoker=3)": "smoking",
    "SKID_Diagnoses": "skid_diagnoses",
    "SKID_Diagnoses 1": "skid_diagnoses_1",
    "SKID_Diagnoses 2": "skid_diagnoses_2",
    "Comments_SKID_assessment": "comments_skid_assessment",
    "Hamilton_Scale": "hamilton_scale",
    "BSL23_sumscore": "bsl23_sumscore",
    "BSL23_behavior": "bsl23_behavior",
    "AUDIT": "audit",
    "Standard_Alcoholunits_Last_28days": "standard_alcoholunits_last_28days",
    "Alcohol_Dependence_In_1st-3rd_Degree_relative": "alcohol_dependence_in_1st_3rd_degree_relative",
    "Relationship_Status": "relationship_status"
})


In [261]:
X = df_dataset[NODE_FEATURES]

# remover linhas com NaN mantendo alinhamento
valid_idx = X.dropna().index
X_clean = X_clean.loc[valid_idx].reset_index(drop=True)
meta_clean = meta_clean.loc[valid_idx].reset_index(drop=True)

X_clean.shape



(0, 366)

In [262]:


# escalar
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clean)

# PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# dataframe final
pca_df = pd.DataFrame(X_pca, columns=["PC1", "PC2"])
pca_df["condition"] = meta_clean["condition"]
pca_df["gender"] = meta_clean["gender"]
pca_df["age"] = meta_clean["age"]


ValueError: Found array with 0 sample(s) (shape=(0, 366)) while a minimum of 1 is required by StandardScaler.

In [ ]:
X_clean.isna().sum().sort_values(ascending=False).head(10)


degree_0    0
degree_1    0
degree_2    0
degree_3    0
degree_4    0
degree_5    0
degree_6    0
degree_7    0
degree_8    0
degree_9    0
dtype: int64

In [ ]:
np.isinf(X_clean).sum().sort_values(ascending=False).head(10)

degree_0    0
degree_1    0
degree_2    0
degree_3    0
degree_4    0
degree_5    0
degree_6    0
degree_7    0
degree_8    0
degree_9    0
dtype: int64

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clean)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(X_pca, columns=["PC1", "PC2"])
pca_df["condition"] = meta_clean["condition"]
pca_df["gender"] = meta_clean["gender"]
pca_df["age"] = meta_clean["age"]


ValueError: Found array with 0 sample(s) (shape=(0, 366)) while a minimum of 1 is required by StandardScaler.

In [ ]:
pca_df

,PC1,PC2,condition,gender,age
0,6.522880,-0.829827,NaN,NaN,NaN
1,3.299071,0.311313,NaN,NaN,NaN
2,-1.637373,-0.590490,NaN,NaN,NaN
3,-7.486036,1.653418,NaN,NaN,NaN
4,6.148981,-0.222944,NaN,NaN,NaN
...,...,...,...,...,...
160,4.202675,-3.347017,NaN,NaN,NaN
161,10.459011,-0.002485,NaN,NaN,NaN
162,2.569105,-0.842226,NaN,NaN,NaN
163,5.635848,-2.042151,NaN,NaN,NaN


In [ ]:

pca_df["condition"] = meta_clean["condition"]
pca_df["gender"] = meta_clean["gender"]
pca_df["age"] = meta_clean["age"]

fig = px.scatter(
    pca_df,
    x="PC1",
    y="PC2",
    color="gender",
    opacity=0.8,
    title="PCA das Métricas de Grau (Nós)"
)
fig.show()


In [ ]:
pca_df

NameError: name 'pca_df' is not defined

In [ ]:
pca_3 = PCA(n_components=2)
pca_3.fit(df_dataset[NODE_FEATURES])

X_pca_3 = pca_3.transform(df_dataset[NODE_FEATURES])

df_pca_3 = pd.DataFrame(X_pca_3, columns=["PC1", "PC2"])
df_pca_3["age"] = df_dataset["age"]
df_pca_3["gender"] = df_dataset["gender"]
df_pca_3["condition"] = df_dataset["condition"]

fig = px.scatter(df_pca_3, x="PC1", y="PC2", color="gender", opacity=0.8)
fig.show()


ValueError: Input X contains NaN.
PCA does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

### Contribuição das features

In [ ]:
loadings = pd.DataFrame(
    pca_2.components_.T,
    index=NODE_FEATURES,
    columns=["PC1", "PC2"]
)


In [ ]:
loadings

,PC1,PC2
degree_0,0.077675,-0.034629
degree_1,0.076115,-0.036981
degree_2,0.078581,-0.044435
degree_3,0.073001,-0.053006
degree_4,0.070616,-0.054444
...,...,...
weighted_degree_56,0.050196,-0.067618
weighted_degree_57,0.050254,-0.091723
weighted_degree_58,0.042000,-0.108827
weighted_degree_59,0.030978,-0.142661


In [ ]:
loadings.abs().sort_values("PC1", ascending=False).head(10)

,PC1,PC2
weighted_degree_46,0.082558,0.017579
degree_46,0.082558,0.017579
weighted_degree_16,0.082146,0.031428
degree_16,0.082146,0.031428
weighted_degree_6,0.082016,0.028846
degree_6,0.082016,0.028846
weighted_degree_10,0.081638,0.019646
degree_10,0.081638,0.019646
weighted_degree_52,0.081264,0.048791
degree_52,0.081264,0.048791


In [ ]:
loadings.abs().sort_values("PC2", ascending=False).head(10)

,PC1,PC2
clustering_20,0.056906,0.150630
local_efficiency_59,0.025072,0.142863
degree_59,0.030978,0.142661
weighted_degree_59,0.030978,0.142661
clustering_59,0.031085,0.135050
local_efficiency_60,0.021359,0.124430
degree_60,0.025563,0.122206
weighted_degree_60,0.025563,0.122206
clustering_40,0.070889,0.120346
clustering_15,0.071363,0.120328


Quais métricas e nós mais contribuem para cada componente?

### PCA com mais componentes - Para ML

In [ ]:
pca_n = PCA(n_components=0.9)  # mantém 90% da variância
X_pca_n = pca_n.fit_transform(X_scaled)

X_pca_n.shape

(165, 29)

A aplicação do PCA sobre as métricas locais permitiu reduzir significativamente
a dimensionalidade do conjunto de dados, preservando a maior parte da variância
original.

A projeção bidimensional indica padrões de organização distintos entre as
condições experimentais, embora com sobreposição parcial, sugerindo que métodos
não lineares ou classificadores supervisionados podem capturar diferenças mais sutis.

Os loadings indicam que métricas associadas a determinados nós apresentam maior
contribuição para os componentes principais, reforçando a relevância da análise
local da conectividade funcional.
